# Tesi QML — Colab quickstart

Verifica il bundle PennyLane e gestisce l'esecuzione sia della matrice primaria sul dataset full (160 run) sia dell'analisi secondaria riprendibile sui subset 100, 300 e 900 (480 run). Prima di iniziare, carica `colab_code.tar.gz` e `colab_dataset.tar.gz` nella cartella Drive `MyDrive/tesi_qml_colab`. Al primo avvio la cella di installazione riavvia automaticamente il kernel: attendi la riconnessione, poi riesegui dalla prima cella; il secondo passaggio salta `pip` ed evita il runtime NumPy misto.

Tutta la logica scientifica, di esecuzione e di salvataggio è delegata agli script di progetto (`scripts/colab_session.py`, `scripts/colab_subsets.py`, ecc.). L'aggregazione finale e la produzione delle figure sono unificate in un'unica cella al termine del notebook.

In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/tesi_qml_colab")
PROJECT_ROOT = Path("/content/tesi_finale")
CODE_ARCHIVE = DRIVE_ROOT / "colab_code.tar.gz"
DATASET_ARCHIVE = DRIVE_ROOT / "colab_dataset.tar.gz"
STATE_ARCHIVE = DRIVE_ROOT / "colab_state.tar.gz"
SUBSET_STATE_ARCHIVE = DRIVE_ROOT / "colab_subset_state.tar.gz"


def run(command):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)


for archive in (CODE_ARCHIVE, DATASET_ARCHIVE):
    if not archive.is_file():
        raise FileNotFoundError(f"Carica prima su Drive: {archive}")
print({"drive_root": str(DRIVE_ROOT), "previous_state": STATE_ARCHIVE.is_file()})

In [ ]:
import hashlib
import os
import shutil
import signal
import sys

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)
run(["tar", "-xzf", CODE_ARCHIVE, "-C", PROJECT_ROOT])
run(["tar", "-xzf", DATASET_ARCHIVE, "-C", PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
if not (PROJECT_ROOT / "requirements-colab.txt").is_file():
    shutil.copy2(
        PROJECT_ROOT / "requirements-cluster.txt",
        PROJECT_ROOT / "requirements-colab.txt",
    )
requirements_path = PROJECT_ROOT / "requirements-cluster.txt"
requirements_fingerprint = hashlib.sha256(
    sys.version.encode("utf-8") + b"\0" + requirements_path.read_bytes()
).hexdigest()
ENV_READY_MARKER = Path("/content/.tesi_qml_environment_ready")
installed_fingerprint = (
    ENV_READY_MARKER.read_text(encoding="utf-8").strip() if ENV_READY_MARKER.is_file() else None
)
if installed_fingerprint != requirements_fingerprint:
    run(["python", "-m", "pip", "install", "-q", "-r", "requirements-cluster.txt"])
    run(["python", "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
    ENV_READY_MARKER.write_text(requirements_fingerprint, encoding="utf-8")
    print(
        "Dipendenze installate. Riavvio automatico del kernel: attendi la riconnessione e riesegui dalla prima cella.",
        flush=True,
    )
    os.kill(os.getpid(), signal.SIGKILL)
print("Ambiente già installato: pip saltato; import eseguiti in un kernel pulito.")
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print({"device": DEVICE, "project": str(PROJECT_ROOT)})

In [ ]:
run(
    [
        "python",
        "scripts/check_storage.py",
        "--project-root",
        PROJECT_ROOT,
        "--min-free-gb",
        1,
        "--min-free-inodes",
        1000,
    ]
)
run(["python", "-m", "pytest", "-m", "not quantum", "-q"])
run(["python", "-m", "pytest", "tests/quantum/test_pennylane_qnn.py", "-q"])
run(
    [
        "python",
        "scripts/colab_session.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        STATE_ARCHIVE,
        "restore",
    ]
)
run(
    [
        "python",
        "-m",
        "pytest",
        "tests/test_matrix.py::test_primary_matrix_has_exactly_160_unique_iteration_one_runs",
        "-q",
    ]
)

## 1. Primo run di prova

Questa cella prepara la feature bank del primo seed se mancante, esegue il primo run pending della matrice primaria e salva immediatamente risultati e feature su Drive.

In [ ]:
run(
    [
        "python",
        "scripts/colab_session.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        STATE_ARCHIVE,
        "run",
        "--max-runs",
        1,
        "--device",
        DEVICE,
    ]
)
run(
    [
        "python",
        "scripts/colab_session.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        STATE_ARCHIVE,
        "status",
    ]
)

## 2. Esecuzione matrice primaria full (160 run)

Esegue in modo incrementale e riprendibile la campagna primaria sul dataset completo (160 run: 2 task × 10 seed × 8 modelli con 5.628 galassie totali). Puoi impostare `FULL_RUN_LIMIT` al numero di run da eseguire nella sessione (es. 160 per l'intera matrice o un valore inferiore per sessioni parziali); dopo ciascun run lo stato viene salvato atomicamente su Drive.

In [ ]:
FULL_RUN_LIMIT = 160
run(
    [
        "python",
        "scripts/colab_session.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        STATE_ARCHIVE,
        "run",
        "--max-runs",
        FULL_RUN_LIMIT,
        "--device",
        DEVICE,
    ]
)
run(
    [
        "python",
        "scripts/colab_session.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        STATE_ARCHIVE,
        "status",
    ]
)

## 3. Analisi secondaria sui subset (480 run)

La matrice secondaria comprende 480 run: 2 task × 10 seed × 3 subset × 8 configurazioni (100, 300, 900 campioni). Validation e test restano identici al protocollo full; i campioni 100, 300 e 900 sono nidificati e condivisi da tutti i modelli. Lo stato incrementale viene gestito da `scripts/colab_subsets.py` in `colab_subset_state.tar.gz`.

In [ ]:
run(
    [
        "python",
        "scripts/colab_subsets.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        SUBSET_STATE_ARCHIVE,
        "prepare",
    ]
)

### Esecuzioni subset separate e riprendibili

Esegui una cella per volta. Ogni subset costituisce un blocco indipendente da 160 run; lo stato viene mostrato e salvato su Drive ogni 10 run completate e in caso di interruzione. Le run già complete vengono saltate automaticamente.

In [ ]:
run(
    [
        "python",
        "scripts/colab_subsets.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        SUBSET_STATE_ARCHIVE,
        "run",
        "--subset",
        "100",
        "--max-runs",
        160,
        "--device",
        DEVICE,
    ]
)

In [ ]:
run(
    [
        "python",
        "scripts/colab_subsets.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        SUBSET_STATE_ARCHIVE,
        "run",
        "--subset",
        "300",
        "--max-runs",
        160,
        "--device",
        DEVICE,
    ]
)

In [ ]:
run(
    [
        "python",
        "scripts/colab_subsets.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        SUBSET_STATE_ARCHIVE,
        "run",
        "--subset",
        "900",
        "--max-runs",
        160,
        "--device",
        DEVICE,
    ]
)

## 4. Aggregazione generale, statistiche e figure della tesi

Questa cella unificata può essere eseguita in qualsiasi momento (dopo la matrice full e/o dopo qualsiasi subset):
1. Raccoglie tutti i run completati tramite `scripts/aggregate_results.py` e valida le ipotesi primarie Q−B2;
2. Genera tutte le figure della tesi tramite `scripts/generate_figures.py`;
3. Se sono presenti subset completati (100, 300, 900), calcola i confronti appaiati secondari con correzione Holm e la curva di apprendimento `macro_f1_vs_training_samples`;
4. Salva e sincronizza atomicamente su Google Drive sia `colab_state.tar.gz` (matrice primaria full) sia `colab_subset_state.tar.gz` (subset secondari).

In [ ]:
run(
    [
        "python",
        "scripts/colab_subsets.py",
        "--project-root",
        PROJECT_ROOT,
        "--state-archive",
        SUBSET_STATE_ARCHIVE,
        "--full-state-archive",
        STATE_ARCHIVE,
        "finalize",
    ]
)